<!-- SPDX-License-Identifier: Apache-2.0 -->
<!-- Copyright (c) 2025 FlyDSL Project Contributors -->

# Structs

Once you have scalars, you often want to **bundle** them — a pair of accumulators,
a small parameter block, a chunk of shared-memory scratch. FlyDSL provides
`@fx.struct`: a decorator that turns an annotated class into a **frozen, C-layout
value type** whose fields are DSL types.

This notebook covers defining a struct, its memory layout, reading/replacing
fields, and the most common real use — describing a shared-memory block.

In [ ]:
import torch

import flydsl.compiler as flyc
import flydsl.expr as fx
from flydsl.compiler.protocol import dsl_align_of, dsl_size_of
from wurlitzer import pipes


def show_gpu_output(launcher, *args, **kwargs):
    """Run a @flyc.jit launcher and echo its GPU printf output into the notebook."""
    kwargs.setdefault("stream", torch.cuda.Stream())
    with pipes() as (out, _err):
        launcher(*args, **kwargs)
        torch.cuda.synchronize()
    print(out.read(), end="")

## 1. Defining a struct

Annotate fields with DSL types. Fields may be scalars, fixed-size `fx.Array`s,
nested structs, or `fx.Constexpr` (compile-time-only, contributes **no** storage).

In [ ]:
@fx.struct
class Pair:
    a: fx.Int32
    b: fx.Float32


@fx.struct
class WithConst:
    a: fx.Int32
    b: fx.Float32
    n: fx.Constexpr[int]   # compile-time only


@fx.struct
class Scratch:
    buf: fx.Array[fx.Float32, 16]
    count: fx.Int32

## 2. Memory layout

A struct lays out like a C struct: fields in declaration order, padded to each
field's natural alignment; the struct's alignment is the max of its fields'. Query
the computed size/alignment (in bytes) with `dsl_size_of` / `dsl_align_of` — no GPU
needed.

In [ ]:
print(f"Pair       size={dsl_size_of(Pair):>3}  align={dsl_align_of(Pair)}")
print(f"WithConst  size={dsl_size_of(WithConst):>3}  align={dsl_align_of(WithConst)}   (constexpr field adds 0 bytes)")
print(f"Scratch    size={dsl_size_of(Scratch):>3}  align={dsl_align_of(Scratch)}   (16 * f32 + i32)")

`Pair` is `4 + 4 = 8` bytes. `WithConst` is also `8` — the `Constexpr` field exists
only at compile time. `Scratch` is `16*4 + 4 = 68` bytes.

## 3. Constructing, reading, and `.replace()`

Struct values are **frozen**: you read fields with `.field`, and produce a modified
copy with `.replace(field=...)`. Field values are DSL values, so — like any other
DSL computation — we build and inspect them *inside a kernel* and print with
`fx.printf`.

In [ ]:
@fx.struct
class Accum:
    total: fx.Float32
    n: fx.Int32


@flyc.kernel
def struct_demo(x: fx.Int32):
    p = Pair(x, x.to(fx.Float32))            # construct from a runtime value
    fx.printf("p.a={}  p.b={}", p.a, p.b)

    q = p.replace(a=x + fx.Int32(100))       # frozen update -> new value
    fx.printf("q.a={}  q.b={}  (p unchanged: p.a={})", q.a, q.b, p.a)

    acc = Accum(total=fx.Float32(0.0), n=fx.Int32(0))
    acc = acc.replace(total=acc.total + fx.Float32(2.5), n=acc.n + fx.Int32(1))
    fx.printf("acc.total={}  acc.n={}", acc.total, acc.n)


@flyc.jit
def run_struct(x: fx.Int32, stream: fx.Stream = fx.Stream(None)):
    struct_demo(x).launch(grid=(1, 1, 1), block=(1, 1, 1), stream=stream)


show_gpu_output(run_struct, fx.Int32(7))

## 4. The common real use: shared-memory scratch

The biggest payoff for structs is describing a **shared-memory layout**. A struct
with `fx.Array` fields names the LDS scratch a kernel needs, and the size/alignment
you just saw is exactly what the shared-memory allocator reserves.

You can see this pattern in the shipped kernels — e.g. the reduction buffers in
`kernels/softmax_kernel.py` and `kernels/layernorm_kernel.py` are declared as a
`@fx.struct SharedStorage` with `fx.Array[fx.Float32, ...]` fields. Allocating and
viewing that storage inside a kernel uses the layout/shared-memory APIs, which are
the subject of a later notebook — here the point is just that **the struct is how
you describe the block.**

---
**Next:** [`03_universal_ops`](03_universal_ops.ipynb) — moving data with the
target-agnostic `Universal*` atoms, ending in a complete vector-add kernel.